In [1]:
# ============================================================
#  OFFLINE RAG SYSTEM: ML ENGINEER RESUME SHORTLISTING
# ============================================================

!pip install PyPDF2 sentence-transformers faiss-cpu

import os
import PyPDF2
import numpy as np
import faiss
from google.colab import files
from sentence_transformers import SentenceTransformer

print(" Libraries Loaded")

# ------------------------------------------------------------
# Step 1: Upload PDF resumes
# ------------------------------------------------------------

print(" Upload Resumes (PDF)...")
uploaded = files.upload()

resume_texts = {}

def extract_text_from_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

for filename in uploaded.keys():
    text = extract_text_from_pdf(filename)
    resume_texts[filename] = text

print(f"📄 Extracted {len(resume_texts)} resumes")


# ------------------------------------------------------------
# Step 2: Define ML Engineer Job Description
# ------------------------------------------------------------

job_description = """
We are hiring a Machine Learning Engineer with strong experience in:
- Python, PyTorch, TensorFlow
- NLP, Transformers, RAG, LLMs
- FAISS, vector databases
- Data preprocessing, model training, evaluation
- MLOps: Docker, MLflow, FastAPI, AWS

Looking for candidates with practical ML project experience.
"""


# ------------------------------------------------------------
# Step 3: Load Local Embedding Model (No HF Access Needed)
# ------------------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")   # downloads once locally, no HF token required
print(" Embedding Model Loaded")


# ------------------------------------------------------------
# Step 4: Create Vector Embeddings for Resumes
# ------------------------------------------------------------

resume_embeddings = []
resume_files = list(resume_texts.keys())

for file in resume_files:
    emb = model.encode(resume_texts[file])
    resume_embeddings.append(emb)

resume_embeddings = np.array(resume_embeddings).astype("float32")

print("Resumes Embedded:", resume_embeddings.shape)


# ------------------------------------------------------------
# Step 5: Build FAISS Index
# ------------------------------------------------------------

index = faiss.IndexFlatL2(resume_embeddings.shape[1])
index.add(resume_embeddings)

print(" FAISS index built")


# ------------------------------------------------------------
# Step 6: Embed Job Description and Query FAISS
# ------------------------------------------------------------

job_embedding = model.encode(job_description).astype("float32")

k = len(resume_files)  # return all ranked
distances, indices = index.search(np.array([job_embedding]), k)

print(" Searching best candidates...\n")


# ------------------------------------------------------------
# Step 7: Shortlist Resumes Based on Similarity
# ------------------------------------------------------------

print(" FINAL SHORTLISTED CANDIDATES:")
print("--------------------------------\n")

ranked_results = []
for rank, idx in enumerate(indices[0]):
    resume_name = resume_files[idx]
    score = 1 / (1 + distances[0][rank])
    ranked_results.append((resume_name, score))

# Sort by score descending
ranked_results = sorted(ranked_results, key=lambda x: x[1], reverse=True)

for i, (name, score) in enumerate(ranked_results, 1):
    print(f"{i}. {name} — Similarity Score: {score:.4f}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.6 MB/s eta 0:00:00
 Libraries Loaded
 Upload Resumes (PDF)...


Saving Arjun_Mehta_Resume.pdf to Arjun_Mehta_Resume (1).pdf
Saving Priya_Raman_Resume.pdf to Priya_Raman_Resume (1).pdf
Saving Karthik_Reddy_Resume.pdf to Karthik_Reddy_Resume (1).pdf
Saving Sneha_Iyer_Resume.pdf to Sneha_Iyer_Resume (1).pdf
Saving Rahul_Verma_Resume.pdf to Rahul_Verma_Resume (1).pdf
📄 Extracted 5 resumes


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding Model Loaded
Resumes Embedded: (5, 384)
 FAISS index built
 Searching best candidates...

 FINAL SHORTLISTED CANDIDATES:
--------------------------------

1. Arjun_Mehta_Resume (1).pdf — Similarity Score: 0.5207
2. Rahul_Verma_Resume (1).pdf — Similarity Score: 0.5062
3. Priya_Raman_Resume (1).pdf — Similarity Score: 0.4841
4. Sneha_Iyer_Resume (1).pdf — Similarity Score: 0.4744
5. Karthik_Reddy_Resume (1).pdf — Similarity Score: 0.4503
